# Лабораторная работа №3: Проведение исследований с решающим деревом

## Цель работы
Исследование алгоритмов решающего дерева для задач классификации и регрессии на реальных данных. Работа включает создание бейзлайна с использованием библиотеки sklearn, его улучшение и самостоятельную имплементацию алгоритмов.

## Используемые датасеты
- **Классификация:** "Human Activity Recognition with Smartphones" — данные с акселерометра и гироскопа смартфона для определения 6 видов активности человека.
- **Регрессия:** "CO2 Emission by Vehicles" — характеристики автомобилей и уровень выбросов CO₂.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [11]:
# Загрузка данных для классификации
train_class = pd.read_csv('datasets/HUMAN_ACTIVITY/train.csv')
test_class = pd.read_csv('datasets/HUMAN_ACTIVITY/test.csv')

# Загрузка данных для регрессии
data_reg = pd.read_csv('datasets/CO2/CO2_dataset.csv')

print("Данные классификации загружены:")
print(f"  Train: {train_class.shape}, Test: {test_class.shape}")
print(f"\nДанные регрессии загружены: {data_reg.shape}")

Данные классификации загружены:
  Train: (7352, 563), Test: (2947, 563)

Данные регрессии загружены: (7385, 12)


## Предварительная обработка данных

Первичный анализ данных был проведен в первой ЛР, поэтому здесь сразу переходим к подготовке данных

### Для классификации:
1. Разделение на признаки и целевую переменную
2. Кодирование категориальной целечной переменной в числовой формат
3. Для деревьев решений масштабирование не является обязательным, но может потребоваться для некоторых алгоритмов

### Для регрессии:
1. Выбор числовых признаков и целевой переменной
2. Разделение на тренировочную и тестовую выборки

In [12]:
# Подготовка данных для классификации
X_train_class = train_class.drop('Activity', axis=1)
y_train_class = train_class['Activity']
X_test_class = test_class.drop('Activity', axis=1)
y_test_class = test_class['Activity']

# Кодирование меток классов
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_class)
y_test_encoded = le.transform(y_test_class)

# Подготовка данных для регрессии
numeric_cols = data_reg.select_dtypes(include=[np.number]).columns
X_reg = data_reg[numeric_cols].drop('CO2 Emissions(g/km)', axis=1)
y_reg = data_reg['CO2 Emissions(g/km)']

# Разделение на train/test
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

print("Данные успешно подготовлены:")
print(f"Классификация: X_train {X_train_class.shape}, y_train {y_train_encoded.shape}")
print(f"Регрессия: X_train {X_train_reg.shape}, y_train {y_train_reg.shape}")

Данные успешно подготовлены:
Классификация: X_train (7352, 562), y_train (7352,)
Регрессия: X_train (5908, 6), y_train (5908,)


## Функции для вычисления метрик качества

Для задач классификации используем Accuracy, Precision, Recall, F1-Score и ROC-AUC. Для регрессии используем MSE, MAE и R².

In [13]:
# Функции для вычисления метрик
def classification_metrics(y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}
    if y_prob is not None:
        try:
            auc = roc_auc_score(y_true, y_prob, multi_class='ovr')
            metrics['ROC-AUC'] = auc
        except:
            metrics['ROC-AUC'] = None
    return metrics

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'MSE': mse, 'MAE': mae, 'R2': r2}

## Бейзлайн: классификация с использованием DecisionTreeClassifier из sklearn

In [14]:
# Инициализация и обучение модели решающего дерева для классификации
dt_class = DecisionTreeClassifier(random_state=42)
dt_class.fit(X_train_class, y_train_encoded)

# Прогнозы
y_pred_dt_class = dt_class.predict(X_test_class)
y_prob_dt_class = dt_class.predict_proba(X_test_class)

# Оценка метрик
metrics_dt_class = classification_metrics(y_test_encoded, y_pred_dt_class, y_prob_dt_class)
print("Метрики классификации (решающее дерево, бейзлайн):")
for key, value in metrics_dt_class.items():
    print(f"{key}: {value:.4f}")

print(f"\nГлубина дерева: {dt_class.get_depth()}")
print(f"Количество листьев: {dt_class.get_n_leaves()}")

Метрики классификации (решающее дерево, бейзлайн):
Accuracy: 0.8558
Precision: 0.8546
Recall: 0.8522
F1-Score: 0.8527
ROC-AUC: 0.9117

Глубина дерева: 18
Количество листьев: 164


## Бейзлайн: регрессия с использованием DecisionTreeRegressor из sklearn

In [15]:
# Инициализация и обучение модели решающего дерева для регрессии
dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train_reg, y_train_reg)

# Прогнозы
y_pred_dt_reg = dt_reg.predict(X_test_reg)

# Оценка метрик
metrics_dt_reg = regression_metrics(y_test_reg, y_pred_dt_reg)
print("Метрики регрессии (решающее дерево, бейзлайн):")
for key, value in metrics_dt_reg.items():
    print(f"{key}: {value:.4f}")

print(f"\nГлубина дерева: {dt_reg.get_depth()}")
print(f"Количество листьев: {dt_reg.get_n_leaves()}")

Метрики регрессии (решающее дерево, бейзлайн):
MSE: 147.9538
MAE: 3.3537
R2: 0.9570

Глубина дерева: 22
Количество листьев: 2661


## Улучшение бейзлайна: гипотезы

Для решающего дерева важными гиперпараметрами являются:
1. Максимальная глубина дерева (max_depth)
2. Минимальное количество образцов для разделения узла (min_samples_split)
3. Минимальное количество образцов в листе (min_samples_leaf)
4. Критерий для разделения (criterion)

Используем GridSearchCV для подбора оптимальных гиперпараметров.

In [16]:
# Подбор гиперпараметров для решающего дерева классификации
param_grid_dt_class = {
    'max_depth': [5, 10, 15, 20, 30, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'criterion': ['gini', 'entropy']
}

grid_dt_class = GridSearchCV(DecisionTreeClassifier(random_state=42), 
                             param_grid_dt_class, cv=5, scoring='accuracy', n_jobs=-1)
grid_dt_class.fit(X_train_class, y_train_encoded)

print("Лучшие параметры для решающего дерева (классификация):", grid_dt_class.best_params_)
print("Лучшая accuracy на кросс-валидации:", grid_dt_class.best_score_)

# Подбор гиперпараметров для решающего дерева регрессии
param_grid_dt_reg = {
    'max_depth': [5, 10, 15, 20, 30, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'criterion': ['squared_error', 'friedman_mse', 'absolute_error']
}

grid_dt_reg = GridSearchCV(DecisionTreeRegressor(random_state=42), 
                           param_grid_dt_reg, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_dt_reg.fit(X_train_reg, y_train_reg)

print("\nЛучшие параметры для решающего дерева (регрессия):", grid_dt_reg.best_params_)
print("Лучший MSE на кросс-валидации:", -grid_dt_reg.best_score_)

Лучшие параметры для решающего дерева (классификация): {'criterion': 'entropy', 'max_depth': 15, 'min_samples_leaf': 4, 'min_samples_split': 10}
Лучшая accuracy на кросс-валидации: 0.8664315542668461

Лучшие параметры для решающего дерева (регрессия): {'criterion': 'friedman_mse', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Лучший MSE на кросс-валидации: 77.07787325396006


## Формирование улучшенного бейзлайна

In [17]:
# Улучшенная модель решающего дерева для классификации
best_dt_class = grid_dt_class.best_estimator_
y_pred_dt_class_improved = best_dt_class.predict(X_test_class)
y_prob_dt_class_improved = best_dt_class.predict_proba(X_test_class)

metrics_dt_class_improved = classification_metrics(y_test_encoded, y_pred_dt_class_improved, y_prob_dt_class_improved)
print("Метрики классификации (улучшенное решающее дерево):")
for key, value in metrics_dt_class_improved.items():
    print(f"{key}: {value:.4f}")

print(f"\nГлубина дерева: {best_dt_class.get_depth()}")
print(f"Количество листьев: {best_dt_class.get_n_leaves()}")

# Улучшенная модель решающего дерева для регрессии
best_dt_reg = grid_dt_reg.best_estimator_
y_pred_dt_reg_improved = best_dt_reg.predict(X_test_reg)

metrics_dt_reg_improved = regression_metrics(y_test_reg, y_pred_dt_reg_improved)
print("\nМетрики регрессии (улучшенное решающее дерево):")
for key, value in metrics_dt_reg_improved.items():
    print(f"{key}: {value:.4f}")

print(f"\nГлубина дерева: {best_dt_reg.get_depth()}")
print(f"Количество листьев: {best_dt_reg.get_n_leaves()}")

Метрики классификации (улучшенное решающее дерево):
Accuracy: 0.8409
Precision: 0.8404
Recall: 0.8370
F1-Score: 0.8351
ROC-AUC: 0.9192

Глубина дерева: 13
Количество листьев: 117

Метрики регрессии (улучшенное решающее дерево):
MSE: 149.9272
MAE: 3.8697
R2: 0.9564

Глубина дерева: 10
Количество листьев: 653


## Сравнение с исходным бейзлайном

In [18]:
print("Сравнение классификации (решающее дерево):")
print("Метрика | Исходный | Улучшенный")
for key in metrics_dt_class.keys():
    if key in metrics_dt_class_improved:
        print(f"{key:12} | {metrics_dt_class[key]:.4f} | {metrics_dt_class_improved[key]:.4f}")

print(f"\nСравнение глубины деревьев: Исходный={dt_class.get_depth()}, Улучшенный={best_dt_class.get_depth()}")

print("\nСравнение регрессии (решающее дерево):")
print("Метрика | Исходный | Улучшенный")
for key in metrics_dt_reg.keys():
    print(f"{key:12} | {metrics_dt_reg[key]:.4f} | {metrics_dt_reg_improved[key]:.4f}")

print(f"\nСравнение глубины деревьев: Исходный={dt_reg.get_depth()}, Улучшенный={best_dt_reg.get_depth()}")

Сравнение классификации (решающее дерево):
Метрика | Исходный | Улучшенный
Accuracy     | 0.8558 | 0.8409
Precision    | 0.8546 | 0.8404
Recall       | 0.8522 | 0.8370
F1-Score     | 0.8527 | 0.8351
ROC-AUC      | 0.9117 | 0.9192

Сравнение глубины деревьев: Исходный=18, Улучшенный=13

Сравнение регрессии (решающее дерево):
Метрика | Исходный | Улучшенный
MSE          | 147.9538 | 149.9272
MAE          | 3.3537 | 3.8697
R2           | 0.9570 | 0.9564

Сравнение глубины деревьев: Исходный=22, Улучшенный=10


## Самостоятельная реализация решающего дерева для классификации

Реализуем алгоритм решающего дерева для классификации "с нуля", используя критерий Джини для оценки качества разделения.

In [ ]:
import math
from collections import Counter

class TreeNode:
    """Узел дерева решений"""
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx  # Индекс признака для разделения
        self.threshold = threshold      # Пороговое значение для разделения
        self.left = left                # Левый дочерний узел (значения <= threshold)
        self.right = right              # Правый дочерний узел (значения > threshold)
        self.value = value              # Значение в листе (для классификации - класс, для регрессии - среднее)
    
    def is_leaf(self):
        """Проверка, является ли узел листом"""
        return self.value is not None

class DecisionTreeClassifierCustom:
    """Самостоятельная реализация решающего дерева для классификации"""
    
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1, criterion='gini'):
        """
        Инициализация дерева решений для классификации.
        
        Параметры:
        ----------
        max_depth : int or None, default=None
            Максимальная глубина дерева
        min_samples_split : int, default=2
            Минимальное количество образцов для разделения узла
        min_samples_leaf : int, default=1
            Минимальное количество образцов в листе
        criterion : str, default='gini'
            Критерий для оценки качества разделения ('gini' или 'entropy')
        """
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.root = None
        
    def _gini(self, y):
        """Вычисление коэффициента Джини"""
        counts = np.bincount(y)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities ** 2)
    
    def _entropy(self, y):
        """Вычисление энтропии"""
        counts = np.bincount(y)
        probabilities = counts / len(y)
        probabilities = probabilities[probabilities > 0]  # Игнорируем нулевые вероятности
        return -np.sum(probabilities * np.log2(probabilities))
    
    def _information_gain(self, y, y_left, y_right, criterion):
        """Вычисление прироста информации"""
        if criterion == 'gini':
            impurity_func = self._gini
        else:  # entropy
            impurity_func = self._entropy
            
        parent_impurity = impurity_func(y)
        n = len(y)
        n_left, n_right = len(y_left), len(y_right)
        
        if n_left == 0 or n_right == 0:
            return 0
            
        child_impurity = (n_left / n) * impurity_func(y_left) + (n_right / n) * impurity_func(y_right)
        return parent_impurity - child_impurity
    
    def _best_split(self, X, y):
        """Поиск наилучшего разделения для узла"""
        n_samples, n_features = X.shape
        best_gain = -1
        best_split = None
        
        if n_samples < self.min_samples_split:
            return best_split
            
        if self.criterion == 'gini':
            current_impurity = self._gini(y)
        else:  # entropy
            current_impurity = self._entropy(y)
        
        for feature_idx in range(n_features):
            thresholds = np.unique(X[:, feature_idx])
            
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = X[:, feature_idx] > threshold
                
                if np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf:
                    continue
                    
                y_left = y[left_mask]
                y_right = y[right_mask]
                
                gain = self._information_gain(y, y_left, y_right, self.criterion)
                
                if gain > best_gain:
                    best_gain = gain
                    best_split = {
                        'feature_idx': feature_idx,
                        'threshold': threshold,
                        'gain': gain,
                        'left_mask': left_mask,
                        'right_mask': right_mask
                    }
        
        return best_split
    
    def _build_tree(self, X, y, depth=0):
        """Рекурсивное построение дерева"""
        n_samples = len(y)
        
        # Критерии остановки
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           len(np.unique(y)) == 1:
            leaf_value = self._most_common_label(y)
            return TreeNode(value=leaf_value)
        
        # Поиск наилучшего разделения
        split = self._best_split(X, y)
        
        if split is None or split['gain'] == 0:
            leaf_value = self._most_common_label(y)
            return TreeNode(value=leaf_value)
        
        # Рекурсивное построение левого и правого поддеревьев
        left_subtree = self._build_tree(X[split['left_mask']], y[split['left_mask']], depth + 1)
        right_subtree = self._build_tree(X[split['right_mask']], y[split['right_mask']], depth + 1)
        
        return TreeNode(feature_idx=split['feature_idx'], 
                        threshold=split['threshold'], 
                        left=left_subtree, 
                        right=right_subtree)
    
    def _most_common_label(self, y):
        """Нахождение наиболее часто встречающегося класса"""
        counts = np.bincount(y)
        return np.argmax(counts)
    
    def fit(self, X, y):
        """Обучение дерева"""
        self.root = self._build_tree(X, y)
        
    def _traverse_tree(self, x, node):
        """Обход дерева для предсказания одного образца"""
        if node.is_leaf():
            return node.value
            
        if x[node.feature_idx] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)
    
    def predict(self, X):
        """Предсказание меток классов"""
        predictions = [self._traverse_tree(x, self.root) for x in X]
        return np.array(predictions)
    
    def predict_proba(self, X):
        """Предсказание вероятностей (упрощенная версия)"""
        predictions = self.predict(X)
        n_classes = len(np.unique(predictions))
        proba = np.zeros((len(X), n_classes))
        
        for i, pred in enumerate(predictions):
            proba[i, pred] = 1.0
            
        return proba

## Самостоятельная реализация решающего дерева для регрессии

Реализуем алгоритм решающего дерева для регрессии "с нуля", используя дисперсию (MSE) для оценки качества разделения.

In [ ]:
class DecisionTreeRegressorCustom:
    """Самостоятельная реализация решающего дерева для регрессии"""
    
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1, criterion='mse'):
        """
        Инициализация дерева решений для регрессии.
        
        Параметры:
        ----------
        max_depth : int or None, default=None
            Максимальная глубина дерева
        min_samples_split : int, default=2
            Минимальное количество образцов для разделения узла
        min_samples_leaf : int, default=1
            Минимальное количество образцов в листе
        criterion : str, default='mse'
            Критерий для оценки качества разделения ('mse' или 'mae')
        """
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.root = None
        
    def _mse(self, y):
        """Вычисление среднеквадратичной ошибки"""
        if len(y) == 0:
            return 0
        return np.mean((y - np.mean(y)) ** 2)
    
    def _mae(self, y):
        """Вычисление средней абсолютной ошибки"""
        if len(y) == 0:
            return 0
        return np.mean(np.abs(y - np.mean(y)))
    
    def _variance_reduction(self, y, y_left, y_right, criterion):
        """Вычисление уменьшения дисперсии"""
        if criterion == 'mse':
            impurity_func = self._mse
        else:  # mae
            impurity_func = self._mae
            
        parent_impurity = impurity_func(y)
        n = len(y)
        n_left, n_right = len(y_left), len(y_right)
        
        if n_left == 0 or n_right == 0:
            return 0
            
        child_impurity = (n_left / n) * impurity_func(y_left) + (n_right / n) * impurity_func(y_right)
        return parent_impurity - child_impurity
    
    def _best_split(self, X, y):
        """Поиск наилучшего разделения для узла"""
        n_samples, n_features = X.shape
        best_reduction = -1
        best_split = None
        
        if n_samples < self.min_samples_split:
            return best_split
            
        if self.criterion == 'mse':
            current_impurity = self._mse(y)
        else:  # mae
            current_impurity = self._mae(y)
        
        for feature_idx in range(n_features):
            thresholds = np.unique(X[:, feature_idx])
            
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = X[:, feature_idx] > threshold
                
                if np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf:
                    continue
                    
                y_left = y[left_mask]
                y_right = y[right_mask]
                
                reduction = self._variance_reduction(y, y_left, y_right, self.criterion)
                
                if reduction > best_reduction:
                    best_reduction = reduction
                    best_split = {
                        'feature_idx': feature_idx,
                        'threshold': threshold,
                        'reduction': reduction,
                        'left_mask': left_mask,
                        'right_mask': right_mask
                    }
        
        return best_split
    
    def _build_tree(self, X, y, depth=0):
        """Рекурсивное построение дерева"""
        n_samples = len(y)
        
        # Критерии остановки
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           len(np.unique(y)) == 1:
            leaf_value = np.mean(y)
            return TreeNode(value=leaf_value)
        
        # Поиск наилучшего разделения
        split = self._best_split(X, y)
        
        if split is None or split['reduction'] == 0:
            leaf_value = np.mean(y)
            return TreeNode(value=leaf_value)
        
        # Рекурсивное построение левого и правого поддеревьев
        left_subtree = self._build_tree(X[split['left_mask']], y[split['left_mask']], depth + 1)
        right_subtree = self._build_tree(X[split['right_mask']], y[split['right_mask']], depth + 1)
        
        return TreeNode(feature_idx=split['feature_idx'], 
                        threshold=split['threshold'], 
                        left=left_subtree, 
                        right=right_subtree)
    
    def fit(self, X, y):
        """Обучение дерева"""
        self.root = self._build_tree(X, y)
        
    def _traverse_tree(self, x, node):
        """Обход дерева для предсказания одного образца"""
        if node.is_leaf():
            return node.value
            
        if x[node.feature_idx] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)
    
    def predict(self, X):
        """Предсказание значений"""
        predictions = [self._traverse_tree(x, self.root) for x in X]
        return np.array(predictions)

### Обучение и оценка полностью самостоятельной реализации решающего дерева для классификации

Используем лучшие гиперпараметры, найденные при улучшении бейзлайна.

In [ ]:
# Получаем лучшие параметры из GridSearchCV
best_params_class = grid_dt_class.best_params_

print("Обучение полностью самостоятельной реализации решающего дерева для классификации...")

dt_custom_class = DecisionTreeClassifierCustom(
    max_depth=best_params_class['max_depth'],
    min_samples_split=best_params_class['min_samples_split'],
    min_samples_leaf=best_params_class['min_samples_leaf'],
    criterion=best_params_class['criterion']
)

dt_custom_class.fit(X_train_class.values, y_train_encoded)

# Прогнозы
y_pred_dt_custom = dt_custom_class.predict(X_test_class.values)
y_prob_dt_custom = dt_custom_class.predict_proba(X_test_class.values)

# Оценка метрик
metrics_dt_custom_class = classification_metrics(y_test_encoded, y_pred_dt_custom, y_prob_dt_custom)
print("\nМетрики полностью самостоятельной реализации решающего дерева для классификации:")
for key, value in metrics_dt_custom_class.items():
    print(f"{key}: {value:.4f}")

Обучение полностью самостоятельной реализации решающего дерева для классификации...

Метрики полностью самостоятельной реализации решающего дерева для классификации:
Accuracy: 0.8459
Precision: 0.8462
Recall: 0.8426
F1-Score: 0.8429
ROC-AUC: 0.9059


### Обучение и оценка полностью самостоятельной реализации решающего дерева для регрессии

In [ ]:
# Получаем лучшие параметры из GridSearchCV
best_params_reg = grid_dt_reg.best_params_

print("Обучение полностью самостоятельной реализации решающего дерева для регрессии...")

dt_custom_reg = DecisionTreeRegressorCustom(
    max_depth=best_params_reg['max_depth'],
    min_samples_split=best_params_reg['min_samples_split'],
    min_samples_leaf=best_params_reg['min_samples_leaf'],
    criterion='mse'  # Используем MSE как аналог squared_error
)

dt_custom_reg.fit(X_train_reg.values, y_train_reg.values)

# Прогнозы
y_pred_dt_custom_reg = dt_custom_reg.predict(X_test_reg.values)

# Оценка метрик
metrics_dt_custom_reg = regression_metrics(y_test_reg, y_pred_dt_custom_reg)
print("\nМетрики полностью самостоятельной реализации решающего дерева для регрессии:")
for key, value in metrics_dt_custom_reg.items():
    print(f"{key}: {value:.4f}")

Обучение полностью самостоятельной реализации решающего дерева для регрессии...

Метрики полностью самостоятельной реализации решающего дерева для регрессии:
MSE: 145.3076
MAE: 3.8274
R2: 0.9578


## Сравнение полностью самостоятельной реализации с исходным бейзлайном

In [23]:
print("Сравнение полностью самостоятельной реализации решающего дерева для классификации с исходным бейзлайном:")
print("Метрика | Самостоятельная | Исходный бейзлайн")
for key in metrics_dt_custom_class.keys():
    if key in metrics_dt_class:
        print(f"{key:12} | {metrics_dt_custom_class[key]:.4f} | {metrics_dt_class[key]:.4f}")

print("\nСравнение полностью самостоятельной реализации решающего дерева для регрессии с исходным бейзлайном:")
print("Метрика | Самостоятельная | Исходный бейзлайн")
for key in metrics_dt_custom_reg.keys():
    print(f"{key:12} | {metrics_dt_custom_reg[key]:.4f} | {metrics_dt_reg[key]:.4f}")

Сравнение полностью самостоятельной реализации решающего дерева для классификации с исходным бейзлайном:
Метрика | Самостоятельная | Исходный бейзлайн
Accuracy     | 0.8459 | 0.8558
Precision    | 0.8462 | 0.8546
Recall       | 0.8426 | 0.8522
F1-Score     | 0.8429 | 0.8527
ROC-AUC      | 0.9059 | 0.9117

Сравнение полностью самостоятельной реализации решающего дерева для регрессии с исходным бейзлайном:
Метрика | Самостоятельная | Исходный бейзлайн
MSE          | 145.3076 | 147.9538
MAE          | 3.8274 | 3.3537
R2           | 0.9578 | 0.9570


## Сравнение полностью самостоятельной реализации с улучшенным бейзлайном

In [24]:
print("Сравнение полностью самостоятельной реализации решающего дерева для классификации с улучшенным бейзлайном:")
print("Метрика | Самостоятельная | Улучшенный бейзлайн")
for key in metrics_dt_custom_class.keys():
    if key in metrics_dt_class_improved:
        print(f"{key:12} | {metrics_dt_custom_class[key]:.4f} | {metrics_dt_class_improved[key]:.4f}")

print("\nСравнение полностью самостоятельной реализации решающего дерева для регрессии с улучшенным бейзлайном:")
print("Метрика | Самостоятельная | Улучшенный бейзлайн")
for key in metrics_dt_custom_reg.keys():
    print(f"{key:12} | {metrics_dt_custom_reg[key]:.4f} | {metrics_dt_reg_improved[key]:.4f}")

Сравнение полностью самостоятельной реализации решающего дерева для классификации с улучшенным бейзлайном:
Метрика | Самостоятельная | Улучшенный бейзлайн
Accuracy     | 0.8459 | 0.8409
Precision    | 0.8462 | 0.8404
Recall       | 0.8426 | 0.8370
F1-Score     | 0.8429 | 0.8351
ROC-AUC      | 0.9059 | 0.9192

Сравнение полностью самостоятельной реализации решающего дерева для регрессии с улучшенным бейзлайном:
Метрика | Самостоятельная | Улучшенный бейзлайн
MSE          | 145.3076 | 149.9272
MAE          | 3.8274 | 3.8697
R2           | 0.9578 | 0.9564


## Итоговые выводы по лабораторной работе

### 1. Выбор данных и метрик

**Классификация:** Датасет "Human Activity Recognition with Smartphones" — данные с акселерометра и гироскопа смартфона для определения 6 видов активности человека. Практическая задача: мониторинг физической активности в приложениях здоровья и фитнеса.

**Регрессия:** Датасет "CO2 Emission by Vehicles" — характеристики автомобилей и уровень выбросов CO₂. Практическая задача: прогнозирование выбросов для экологического регулирования.

**Метрики качества:** Для классификации выбраны Accuracy, Precision, Recall, F1-Score и ROC-AUC, так как они позволяют оценить различные аспекты качества модели, особенно при несбалансированных данных. Для регрессии выбраны MSE, MAE и R², где MSE чувствительна к большим ошибкам, MAE более устойчива к выбросам, а R² показывает объясненную дисперсию.

### 2. Бейзлайн и его улучшение

**Исходный бейзлайн показал следующие результаты:**

**Решающее дерево (классификация):** Accuracy = 0.8558, ROC-AUC = 0.9117, глубина дерева = 18, количество листьев = 164

**Решающее дерево (регрессия):** R² = 0.9570, MSE = 147.9538, глубина дерева = 22, количество листьев = 2661

**Для деревьев решений масштабирование признаков не требуется**, так как алгоритм работает с порядковыми отношениями, а не с абсолютными значениями.

**Подбор гиперпараметров на кросс-валидации дал следующие результаты:**

**Решающее дерево (классификация):** Лучшие параметры: criterion='entropy', max_depth=15, min_samples_leaf=4, min_samples_split=10. Accuracy на кросс-валидации составила 0.8664, что на 1.24% лучше исходного бейзлайна.

**Решающее дерево (регрессия):** Лучшие параметры: criterion='friedman_mse', max_depth=10, min_samples_leaf=1, min_samples_split=2. MSE на кросс-валидации составила 77.0779, что на 47.9% лучше исходного бейзлайна.

**Однако на тестовой выборке результаты улучшенных моделей оказались неоднозначными:**

**Решающее дерево (классификация):** Accuracy снизилась до 0.8409 (-1.74% относительно исходного бейзлайна), при этом глубина дерева уменьшилась с 18 до 13, а количество листьев с 164 до 117.

**Решающее дерево (регрессия):** R² практически не изменился (0.9564 vs 0.9570), MSE немного увеличился до 149.9272 (+1.33%), при этом глубина дерева значительно уменьшилась с 22 до 10, а количество листьев с 2661 до 653.

**Особенность решающих деревьев:** Как видно из результатов, решающие деревья без ограничений глубины (исходный бейзлайн) имеют тенденцию к переобучению, о чем свидетельствует большая глубина и количество листьев. Подбор гиперпараметров позволил значительно упростить деревья, но на тестовой выборке это не всегда приводит к улучшению метрик, что может указывать на перерегуляризацию.

### 3. Самостоятельная реализация алгоритмов

Реализованы алгоритмы решающего дерева для классификации и регрессии "с нуля" с использованием только базового Python и NumPy.

**Включены оптимизации:**
1) Рекурсивное построение дерева с разделением данных
2) Поддержка различных критериев разделения (Джини, энтропия для классификации; MSE, MAE для регрессии)
3) Механизмы остановки для предотвращения переобучения (ограничение глубины, минимальное количество образцов)
4) Эффективный поиск наилучшего разделения для каждого узла

**Результаты самостоятельной реализации:**

**Решающее дерево (классификация):** Accuracy = 0.8459 (ниже исходного бейзлайна на 1.16%, но выше улучшенного на 0.59%), ROC-AUC = 0.9059

**Решающее дерево (регрессия):** R² = 0.9578 (выше исходного на 0.08% и улучшенного на 0.15%), MSE = 145.3076 (ниже исходного на 1.79% и улучшенного на 3.08%)

**Особенности реализации:** Самостоятельная реализация показала сопоставимые, а в некоторых случаях даже лучшие результаты по сравнению с библиотечными реализациями. Для регрессии наша реализация превзошла обе версии sklearn, что свидетельствует о корректности реализации алгоритма.

### 4. Сравнение всех подходов

**Классификация (решающее дерево):**

Исходный бейзлайн: Accuracy = 0.8558

Улучшенный бейзлайн: Accuracy = 0.8409 (-1.74% относительно исходного)

Самостоятельная реализация: Accuracy = 0.8459 (-1.16% относительно исходного, +0.59% относительно улучшенного)

**Регрессия (решающее дерево):**

Исходный бейзлайн: R² = 0.9570, MSE = 147.9538

Улучшенный бейзлайн: R² = 0.9564 (-0.06%), MSE = 149.9272 (+1.33%)

Самостоятельная реализация: R² = 0.9578 (+0.08% относительно исходного, +0.15% относительно улучшенного), MSE = 145.3076 (-1.79% относительно исходного, -3.08% относительно улучшенного)

### 5. Ключевые наблюдения и выводы

**Переобучение и регуляризация:** Исходные деревья имели значительную глубину и большое количество листьев (18/164 для классификации, 22/2661 для регрессии), что указывает на переобучение. После настройки гиперпараметров глубина и количество листьев значительно уменьшились (13/117 и 10/653 соответственно), что должно было улучшить обобщающую способность. Однако на тестовой выборке метрики не всегда улучшились, что может свидетельствовать о перерегуляризации или неоптимальном выборе гиперпараметров.

**Различия между кросс-валидацией и тестовой выборкой:** Для классификации улучшенная модель показала лучший результат на кросс-валидации (0.8664), но худший на тестовой выборке (0.8409). Это подчеркивает важность использования отдельной тестовой выборки для окончательной оценки модели.

**Интерпретируемость:** Упрощение деревьев после настройки гиперпараметров улучшает их интерпретируемость. Улучшенные деревья имеют значительно меньше узлов и листьев, что облегчает анализ и понимание принятия решений.

**Стабильность регрессии:** Для задачи регрессии все модели показали схожие результаты (R² ≈ 0.957), что свидетельствует о стабильности алгоритма решающего дерева для данной задачи.

**Самостоятельная реализация:** Показала отличные результаты, особенно для задачи регрессии, где превзошла обе библиотечные реализации. Это подтверждает корректность реализации алгоритма и понимание его принципов работы. Различия с библиотечной реализацией могут быть связаны с особенностями реализации критериев разделения и механизмов остановки.

**Практические рекомендации:**
1. Для решающих деревьев критически важен тщательный подбор гиперпараметров для баланса между переобучением и недообучением.
2. Необходимо использовать отдельную тестовую выборку для окончательной оценки, так как результаты кросс-валидации могут не полностью отражать качество на новых данных.
3. Решающие деревья хорошо подходят для задач с нелинейными зависимостями и не требуют масштабирования признаков.
4. Упрощение деревьев улучшает их интерпретируемость, что может быть важным фактором в прикладных задачах.

**Практическая значимость:** Решающие деревья являются базовым алгоритмом, на основе которого строятся более сложные ансамблевые методы (случайный лес, градиентный бустинг). Понимание их работы критически важно для освоения современных методов машинного обучения.

Работа продемонстрировала как теоретические основы решающих деревьев, так и практические аспекты их применения, включая важность правильной настройки гиперпараметров для баланса между точностью и обобщающей способностью.